In [ ]:
from skrub import GapEncoder
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

trans = pd.read_csv("../data/transactions_2016_2017.csv", low_memory=False)
train = pd.read_csv("../data/customer_clv_train.csv")
features = pd.read_csv("../data/customer_features_lowie1.csv")

# Stap 1: aggregeer tekst naar klantniveau
# prod_title heeft 21.302 unieke waarden — ideaal voor GapEncoder
prod_title_agg = (
    trans.groupby("cust_id")["prod_title"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"prod_title": "prod_titles_concat"})
)

prod_brand_agg = (
    trans.groupby("cust_id")["prod_brand"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={"prod_brand": "prod_brands_concat"})
)

# Stap 2: GapEncoder op geconcateneerde tekst
enc_title = GapEncoder(n_components=10, random_state=42)
enc_brand = GapEncoder(n_components=5, random_state=42)

title_encoded = enc_title.fit_transform(prod_title_agg[["prod_titles_concat"]])
brand_encoded = enc_brand.fit_transform(prod_brand_agg[["prod_brands_concat"]])

title_df = pd.DataFrame(
    title_encoded,
    columns=[f"title_gap_{i}" for i in range(10)],
    index=prod_title_agg["cust_id"]
).reset_index()

brand_df = pd.DataFrame(
    brand_encoded,
    columns=[f"brand_gap_{i}" for i in range(5)],
    index=prod_brand_agg["cust_id"]
).reset_index()

# Stap 3: merge met bestaande features
df = train.merge(features, on="cust_id", how="left")
df = df.merge(title_df, on="cust_id", how="left")
df = df.merge(brand_df, on="cust_id", how="left")
df["returned"] = (df["revenue_2018_2019"] > 0).astype(int)

# Stap 4: test of skrub features AUC verbeteren
feature_cols_base = [c for c in features.columns if c != "cust_id"]
feature_cols_skrub = feature_cols_base + [f"title_gap_{i}" for i in range(10)] + [f"brand_gap_{i}" for i in range(5)]

X = df[feature_cols_skrub].fillna(0)
y = df["returned"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Baseline zonder skrub
model_base = lgb.LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
model_base.fit(X_train[feature_cols_base], y_train)
auc_base = roc_auc_score(y_val, model_base.predict_proba(X_val[feature_cols_base])[:, 1])
print(f"AUC zonder skrub: {auc_base:.4f}")

# Met skrub
model_skrub = lgb.LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
model_skrub.fit(X_train[feature_cols_skrub], y_train)
auc_skrub = roc_auc_score(y_val, model_skrub.predict_proba(X_val[feature_cols_skrub])[:, 1])
print(f"AUC met skrub:    {auc_skrub:.4f}")
print(f"Verschil:         {auc_skrub - auc_base:+.4f}")